In [6]:
import random
from viz import draw_state
import numpy as np
from env import SimpleARGEnvironment
from rollout_worker_arg import RolloutWorker
from tb_gfn import TBGFlowNetGenerator
from utils import load_sequences

In [17]:
Ne = 10000
r_per_bp = 2e-8

dataset_path="../dataset/sim_l1mb_0.fa"
sequences = load_sequences(dataset_path)
sequence_length = len(sequences[0])
num_blocks = sequence_length
rho = 4 * Ne * r_per_bp * num_blocks

In [18]:
env = SimpleARGEnvironment(
        num_sequences=len(sequences),
        sequence_length=sequence_length,
        rho=rho,
        sequences=sequences,
        num_blocks=num_blocks,
        fixed_edge_length=0.02,
        rng=random.Random(7),
        learn_times=True,
    )

# generator = TBGFlowNetGenerator(env, verbose=True)
# rollout_worker = RolloutWorker(env, verbose=True)
# batch_size = 2

In [16]:
import math
from pathlib import Path

import numpy as np
import tskit


def compute_tree_sequence_log_likelihood(ts, env):
    """Score a tskit TreeSequence against env.sequences under JC69."""
    if ts.num_samples != env.num_sequences:
        raise ValueError(
            f"TreeSequence has {ts.num_samples} samples, but env has {env.num_sequences} sequences"
        )
    if int(round(ts.sequence_length)) != env.sequence_length:
        raise ValueError(
            f"TreeSequence length {ts.sequence_length} does not match env length {env.sequence_length}"
        )

    model = env.evolution_model
    prob_floor = model._PROB_FLOOR
    seq_arrays = env.seq_arrays.detach().cpu().numpy().astype(float, copy=False)
    sample_to_seq = {int(sample_id): seq_idx for seq_idx, sample_id in enumerate(ts.samples())}
    transition_cache = {}
    log_likelihood = 0.0

    def branch_length(parent_id, child_id, tree):
        if not env.learn_times:
            return env.fixed_edge_length
        generation_delta = float(tree.time(parent_id) - tree.time(child_id))
        if generation_delta <= 0:
            raise ValueError(
                f"TreeSequence node times must increase from child to parent: "
                f"parent={parent_id}, child={child_id}, delta={generation_delta}"
            )
        return generation_delta * env.mutation_rate

    def transition_matrix(edge_length):
        key = float(edge_length)
        if key not in transition_cache:
            transition_cache[key] = model._jc69_transition_matrix(key)
        return transition_cache[key]

    for tree in ts.trees():
        left, right = tree.interval
        site_start = max(0, int(round(left)))
        site_end = min(env.sequence_length, int(round(right)))
        if site_start >= site_end:
            continue

        width = site_end - site_start
        partials_by_node = {}
        log_scale_by_node = {}

        for node_id in tree.nodes(order="postorder"):
            node_id = int(node_id)
            children = [int(child_id) for child_id in tree.children(node_id)]

            if not children:
                if node_id in sample_to_seq:
                    partials = model._normalize_leaf_partials(
                        seq_arrays[sample_to_seq[node_id], site_start:site_end].copy()
                    )
                else:
                    partials = np.full((width, seq_arrays.shape[-1]), 0.25, dtype=float)
                log_scale = np.zeros(width, dtype=float)
                partials, log_scale = model._rescale_partials(partials, log_scale)
                partials_by_node[node_id] = partials
                log_scale_by_node[node_id] = log_scale
                continue

            partials = np.ones((width, seq_arrays.shape[-1]), dtype=float)
            log_scale = np.zeros(width, dtype=float)
            for child_id in children:
                child_partials = np.maximum(partials_by_node[child_id], prob_floor)
                child_log_scale = log_scale_by_node[child_id]
                trans = transition_matrix(branch_length(node_id, child_id, tree))
                with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
                    partials *= child_partials @ trans.T
                log_scale += child_log_scale
                partials, log_scale = model._rescale_partials(partials, log_scale)

            partials_by_node[node_id] = partials
            log_scale_by_node[node_id] = log_scale

        site_probs = np.ones(width, dtype=float)
        root_log_scale = np.zeros(width, dtype=float)
        for root_id in tree.roots:
            root_id = int(root_id)
            root_partials = partials_by_node[root_id]
            site_probs *= np.maximum(np.sum(root_partials * 0.25, axis=1), prob_floor)
            root_log_scale += log_scale_by_node[root_id]

        with np.errstate(divide="ignore", invalid="ignore"):
            log_likelihood += float(np.log(site_probs).sum() + root_log_scale.sum())

    if not math.isfinite(log_likelihood):
        return model._NON_FINITE_LOG_LIKELIHOOD
    return float(log_likelihood)


truth_trees_path = "./validation/trees/sim_l1mb_0.trees"
truth_ts = tskit.load(str(truth_trees_path))
truth_log_likelihood = compute_tree_sequence_log_likelihood(truth_ts, env)
truth_log_likelihood

-1406773.8556912583

In [19]:
traj = env.sample(1, compute_reward=True)
env.evolution_model.compute_arg_log_likelihood(traj[0])

-1429063.9409014601

In [21]:
-1429063.9409014601 / env.sequence_length

-1.42906394090146

In [ ]:
for state in traj:
    draw_state(state)

In [5]:
env.save_to_tree_sequence(traj[0], "example_time.trees")

In [ ]:
ret, traj = rollout_worker.rollout(generator, episodes=batch_size)
states = ret["states"]
# draw_state(states[0])

In [ ]:
generator.accumulate_loss(ret)

In [ ]:
generator.loss

In [ ]:
last_info = generator.update_model()
# log_z = generator.compute_log_Z().detach().cpu().reshape(-1)[0].item()

In [ ]:
last_info

In [ ]:
draw_state(states[0])

In [ ]:
env.save_to_tree_sequence(states[0], "example.trees")

In [ ]:
act_seg = env.get_arg_sequence_segments(states[0])
[(b['child_node_id'], b['parent_node_ids']) for b in act_seg['recombination_events']]